## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
## add your code here
#include <iostream>
#include <vector>
#include <algorithm>

using namespace std;

// 定义操作指令
struct Command {
    int opType; // 0: 交换, 1: 异或, 2: 模加
    int param;

    static Command swapOp() { return {0, 0}; }
    static Command xorOp(int val) { return {1, val}; }
    static Command addOp(int val) { return {2, val}; }
};

namespace PrefixSolver {
    bool isValidPerm(const vector<int>& arr, int maxVal) {
        vector<bool> visited(maxVal, false);
        for (int item : arr) {
            if (item < 0 || item >= maxVal || visited[item]) return false;
            visited[item] = true;
        }
        return true;
    }

    void insertXor(vector<Command>& seq, int val) {
        if (val) seq.push_back(Command::xorOp(val));
    }

    void insertAdd(vector<Command>& seq, int val) {
        if (val) seq.push_back(Command::addOp(val));
    }

    void compressXor(vector<Command>& seq) {
        vector<Command> res;
        for (const auto& cmd : seq) {
            if (cmd.opType == 1 && !res.empty() && res.back().opType == 1) {
                int combined = res.back().param ^ cmd.param;
                res.pop_back();
                if (combined != 0) res.push_back(Command::xorOp(combined));
            } else {
                if (cmd.opType == 0 || cmd.param != 0) res.push_back(cmd);
            }
        }
        seq = std::move(res);
    }

    bool decompose(const vector<int>& arr, int limit, vector<Command>& operations) {
        operations.clear();
        if (!isValidPerm(arr, limit)) return false;
        if (limit == 1) return true;

        int mid = limit >> 1;
        vector<int> evens(mid), odds(mid);
        
        for (int k = 0; k < mid; ++k) {
            evens[k] = arr[k << 1] >> 1;
            odds[k] = arr[(k << 1) | 1] >> 1;
        }

        vector<Command> lPath, rPath;
        if (!decompose(evens, mid, lPath) || !decompose(odds, mid, rPath)) return false;

        if (arr[0] & 1) {
            if (limit == 2) insertAdd(operations, 1);
            else insertXor(operations, 1);
        }

        int lAcc = 0, rAcc = 0;
        
        for (const auto& cmd : lPath) {
            if (cmd.opType == 2) {
                insertXor(operations, 1);
                insertAdd(operations, 1);
            } else {
                insertXor(operations, cmd.param << 1);
                lAcc ^= (cmd.param << 1);
            }
        }
        insertXor(operations, lAcc);

        for (const auto& cmd : rPath) {
            if (cmd.opType == 2) {
                insertAdd(operations, 1);
                insertXor(operations, 1);
            } else {
                insertXor(operations, cmd.param << 1);
                rAcc ^= (cmd.param << 1);
            }
        }

        if ((lAcc & mid) != (rAcc & mid)) return false;
        
        if (lAcc >= mid) lAcc -= mid;
        if (rAcc >= mid) rAcc -= mid;
        
        if (lAcc != rAcc) return false;

        compressXor(operations);
        return true;
    }
}

class StateEngine {
private:
    int totalSize;
    int anchorU;
    int anchorV;
    int stride;

    vector<int> state;
    vector<Command> logOps;

    bool checkInitialState() const {
        vector<bool> marked(totalSize, false);
        for (int val : state) {
            if (val < 0 || val >= totalSize || marked[val]) return false;
            marked[val] = true;
        }
        return true;
    }

    void execute(const Command& cmd) {
        if ((cmd.opType == 1 || cmd.opType == 2) && cmd.param == 0) return;
        
        logOps.push_back(cmd);

        if (cmd.opType == 0) {
            for (int& val : state) {
                if (val == anchorU) val = anchorV;
                else if (val == anchorV) val = anchorU;
            }
        } else if (cmd.opType == 1) {
            for (int& val : state) val ^= cmd.param;
        } else {
            for (int& val : state) val = (val + cmd.param) % totalSize;
        }
    }

    void shiftAll(int offset) {
        offset %= totalSize;
        if (offset < 0) offset += totalSize;
        execute(Command::addOp(offset));
    }

    void flipAll(int mask) {
        execute(Command::xorOp(mask));
    }

    pair<int, int> calcBridge(int u, int v) const {
        int dist = (v - u + totalSize - stride + totalSize) % totalSize;
        int pt1 = 0, pt2 = 0;

        for (int chunk = totalSize >> 1; chunk >= (stride << 1); chunk >>= 1) {
            if (dist >= chunk) {
                dist -= chunk;
                pt2 += chunk >> 1;
            } else {
                pt1 += chunk >> 1;
            }
        }

        int remainder = u & (stride - 1);
        pt1 += (totalSize >> 1) + remainder;
        pt2 += remainder;

        return {pt1, pt2};
    }

    void swapArbitrary(int u, int v) {
        if (u == v) return;

        int blkU = (u / stride) & 1;
        int blkV = (v / stride) & 1;

        if (blkU == blkV) {
            int midPt = (blkU == 0) ? ((u & (stride - 1)) + stride) : (u & (stride - 1));
            swapArbitrary(u, midPt);
            swapArbitrary(v, midPt);
            swapArbitrary(u, midPt);
            return;
        }

        auto targetPair = calcBridge(anchorU, anchorV);
        auto srcPair = calcBridge(u, v);

        shiftAll(srcPair.first - u);
        flipAll(srcPair.first ^ targetPair.first);
        shiftAll(anchorU - targetPair.first);

        execute(Command::swapOp());

        shiftAll(targetPair.first - anchorU);
        flipAll(srcPair.first ^ targetPair.first);
        shiftAll(u - srcPair.first);
    }

public:
    StateEngine(int sz, int u, int v, const vector<int>& initialData)
        : totalSize(sz), anchorU(u), anchorV(v), state(initialData) {
        int delta = (anchorU - anchorV + totalSize) % totalSize;
        stride = (delta == 0) ? totalSize : (delta & -delta);
    }

    bool run() {
        if (!checkInitialState()) return false;

        if (stride > 1) {
            vector<int> rem(stride);
            for (int k = 0; k < stride; ++k) {
                rem[k] = state[k] & (stride - 1);
            }

            vector<Command> phase1;
            if (!PrefixSolver::decompose(rem, stride, phase1)) return false;

            for (const auto& cmd : phase1) {
                execute(cmd);
            }
        }

        for (int r = 0; r < stride; ++r) {
            vector<int> group;
            for (int curr = r; curr < totalSize; curr += stride) {
                group.push_back(state[curr]);
            }
            sort(group.begin(), group.end());

            int index = 0;
            for (int curr = r; curr < totalSize; curr += stride, ++index) {
                if (group[index] != curr) return false;
            }

            for (int curr = r; curr < totalSize; curr += stride) {
                if (state[curr] != curr) {
                    swapArbitrary(curr, state[curr]);
                }
            }
        }

        for (int k = 0; k < totalSize; ++k) {
            if (state[k] != k) return false;
        }
        return true;
    }

    const vector<Command>& getResult() const {
        return logOps;
    }
};

int main() {
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);

    int m, a, b;
    if (!(cin >> m >> a >> b)) return 0;

    vector<int> arr(m);
    for (int k = 0; k < m; ++k) {
        cin >> arr[k];
    }

    StateEngine engine(m, a, b, arr);

    if (!engine.run()) {
        cout << "-1\n";
    } else {
        const auto& result = engine.getResult();
        cout << result.size() << "\n";
        for (const auto& cmd : result) {
            if (cmd.opType == 0) {
                cout << "0\n";
            } else {
                cout << cmd.opType << " " << cmd.param << "\n";
            }
        }
    }

    return 0;
}

## B 长跑

In [ ]:
## add your code here
import sys

def solve():
    input_data = sys.stdin.read().split()
    if not input_data:
        return
    
    idx = 0
    n_tokens = len(input_data)
    
    while idx < n_tokens:
        if idx + 4 > n_tokens:
            break
            
        n = int(input_data[idx])
        l = int(input_data[idx+1])
        maxn = int(input_data[idx+2])
        s = int(input_data[idx+3])
        idx += 4
        
        stations = []
        for _ in range(n):
            pi = int(input_data[idx])
            ci = int(input_data[idx+1])
            stations.append((pi, ci))
            idx += 2
            
        stations.sort(key=lambda x: x[0])
        
        # dp[j] 表示剩余 j 个硬币时的最大体力，-1 表示不可达
        dp = [-1] * (s + 1)
        dp[s] = maxn
        
        current_pos = 0
        for pi, ci in stations:
            if pi > l:
                continue
                
            # 1. 处理跑到当前补给站的体力消耗
            dist = pi - current_pos
            for j in range(s + 1):
                if dp[j] != -1:
                    dp[j] -= dist
                    if dp[j] < 0:
                        dp[j] = -1 # 体力透支，此状态失效
                        
            current_pos = pi
            
            # 2. 在当前补给站花费硬币补充体力
            for j in range(ci, s + 1):
                if dp[j] != -1:
                    if maxn > dp[j - ci]:
                        dp[j - ci] = maxn
                        
        # 3. 处理从最后一个有效补给站跑到终点 L 的体力消耗
        dist_to_l = l - current_pos
        possible = False
        for j in range(s + 1):
            if dp[j] != -1:
                if dp[j] >= dist_to_l:
                    possible = True
                    break
                    
        if possible:
            print("Yes")
        else:
            print("No")

if __name__ == '__main__':
    solve()

## C 最长回文

In [ ]:
## add your code here
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>

using namespace std;

typedef unsigned long long ull;
const ull BASE = 131;

vector<int> manacher(const string& s) {
    int n = s.length();
    string s2(n * 2 + 1, '#');
    for (int i = 0; i < n; ++i) {
        s2[i * 2 + 1] = s[i];
    }
    
    int m = s2.length();
    vector<int> d(m, 0);
    for (int i = 0, l = 0, r = -1; i < m; ++i) {
        int k = (i > r) ? 1 : min(d[l + r - i], r - i + 1);
        while (i - k >= 0 && i + k < m && s2[i - k] == s2[i + k]) {
            k++;
        }
        d[i] = k;
        k--;
        if (i + k > r) {
            l = i - k;
            r = i + k;
        }
    }
    return d;
}

int main() {
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);

    int n;
    if (!(cin >> n)) return 0;
    
    string A, B;
    cin >> A >> B;

    string RevA = A;
    reverse(RevA.begin(), RevA.end());

    vector<ull> pw(n + 1, 1);
    for (int i = 1; i <= n; ++i) {
        pw[i] = pw[i - 1] * BASE;
    }

    vector<ull> hRevA(n + 1, 0), hB(n + 1, 0);
    for (int i = 0; i < n; ++i) {
        hRevA[i + 1] = hRevA[i] * BASE + RevA[i];
        hB[i + 1] = hB[i] * BASE + B[i];
    }

    // Lambda 函数：获取闭区间 [l, r] 的哈希值
    auto getHashRevA = [&](int l, int r) {
        return hRevA[r + 1] - hRevA[l] * pw[r - l + 1];
    };
    auto getHashB = [&](int l, int r) {
        return hB[r + 1] - hB[l] * pw[r - l + 1];
    };

    vector<int> dA = manacher(A);
    vector<int> dB = manacher(B);

    int ans = 0;

    // 1. 遍历 A 中的所有回文中心
    for (int c = 1; c < 2 * n; ++c) {
        int L = dA[c] - 1;
        if (L == 0) continue;
        
        int P = (c - L) / 2;
        int K = P + L - 1;

        int startA = n - P;
        int startB = K;
        int high = min(n - startA, n - startB);

        if (L + 2 * high <= ans) continue;

        int ans_E = 0;
        int low = 1;
        while (low <= high) {
            int mid = low + (high - low) / 2;
            if (getHashRevA(startA, startA + mid - 1) == getHashB(startB, startB + mid - 1)) {
                ans_E = mid;
                low = mid + 1;
            } else {
                high = mid - 1;
            }
        }
        ans = max(ans, L + 2 * ans_E);
    }

    // 2. 遍历 B 中的所有回文中心
    for (int c = 1; c < 2 * n; ++c) {
        int L = dB[c] - 1;
        if (L == 0) continue;
        
        int P = (c - L) / 2;
        int K = P + L - 1;

        int startA = n - 1 - P;
        int startB = K + 1;
        int high = min(n - startA, n - startB);

        if (L + 2 * high <= ans) continue;

        int ans_E = 0;
        int low = 1;
        while (low <= high) {
            int mid = low + (high - low) / 2;
            if (getHashRevA(startA, startA + mid - 1) == getHashB(startB, startB + mid - 1)) {
                ans_E = mid;
                low = mid + 1;
            } else {
                high = mid - 1;
            }
        }
        ans = max(ans, L + 2 * ans_E);
    }

    // 3. 遍历空回文中心 (中心边界恰好在 A[k] 与 B[k] 之间)
    for (int k = 0; k < n; ++k) {
        int startA = n - 1 - k;
        int startB = k;
        int high = min(n - startA, n - startB);

        if (2 * high <= ans) continue;

        int ans_E = 0;
        int low = 1;
        while (low <= high) {
            int mid = low + (high - low) / 2;
            if (getHashRevA(startA, startA + mid - 1) == getHashB(startB, startB + mid - 1)) {
                ans_E = mid;
                low = mid + 1;
            } else {
                high = mid - 1;
            }
        }
        ans = max(ans, 2 * ans_E);
    }

    cout << ans << "\n";
    return 0;
}

## D 优惠券

In [ ]:
## add your code here
#include <bits/stdc++.h>
using namespace std;

struct BIT {
    int n;
    vector<int> tree;

    void init(int n_) {
        n = n_;
        tree.assign(n + 1, 0);
    }

    void add(int idx, int val) {
        for (; idx <= n; idx += idx & -idx) {
            tree[idx] += val;
        }
    }

    int sum(int idx) {
        if (idx > n) idx = n;
        int res = 0;
        for (; idx > 0; idx -= idx & -idx) {
            res += tree[idx];
        }
        return res;
    }

    int kth(int k) {
        int idx = 0;
        int bit = 1;
        while ((bit << 1) <= n) bit <<= 1;

        for (; bit; bit >>= 1) {
            int nxt = idx + bit;
            if (nxt <= n && tree[nxt] < k) {
                idx = nxt;
                k -= tree[nxt];
            }
        }

        return idx + 1;
    }
};

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    const int MAXX = 100000;

    vector<int> last(MAXX + 1, 0);
    vector<int> touched;

    int m;

    while (cin >> m) {
        BIT bit;
        bit.init(m);

        touched.clear();
        int ans = -1;

        auto use_question_after = [&](int pos) -> bool {
            int before = bit.sum(pos);
            int total = bit.sum(m);

            if (total == before) {
                return false;
            }

            int p = bit.kth(before + 1);
            bit.add(p, -1);
            return true;
        };

        for (int i = 1; i <= m; ++i) {
            string op;
            cin >> op;

            if (op != "I" && op != "O") {
                if (ans == -1) {
                    bit.add(i, 1);
                }
                continue;
            }

            int x;
            cin >> x;

            if (ans != -1) {
                continue;
            }

            if (op == "I") {
                if (last[x] > 0) {
                    if (!use_question_after(last[x])) {
                        ans = i;
                        continue;
                    }
                }

                if (last[x] == 0) {
                    touched.push_back(x);
                }

                last[x] = i;
            } else {
                // op == "O"
                if (last[x] <= 0) {
                    if (!use_question_after(-last[x])) {
                        ans = i;
                        continue;
                    }
                }

                if (last[x] == 0) {
                    touched.push_back(x);
                }

                last[x] = -i;
            }
        }

        cout << ans << '\n';

        for (int x : touched) {
            last[x] = 0;
        }
    }

    return 0;
}

## E 任意点

In [ ]:
## add your code here
#include <iostream>
#include <vector>

using namespace std;

int n;
vector<int> x, y;
vector<bool> visited;

// 深度优先搜索，标记同一个连通块内的所有点
void dfs(int u) {
    visited[u] = true;
    for (int v = 0; v < n; ++v) {
        // 如果点 v 未被访问，且与点 u 在同一行或同一列
        if (!visited[v] && (x[u] == x[v] || y[u] == y[v])) {
            dfs(v);
        }
    }
}

int main() {
    // 优化标准输入输出速度
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);

    if (!(cin >> n)) return 0;

    x.resize(n);
    y.resize(n);
    visited.assign(n, false);

    for (int i = 0; i < n; ++i) {
        cin >> x[i] >> y[i];
    }

    int components = 0; // 记录连通块的数量
    for (int i = 0; i < n; ++i) {
        if (!visited[i]) {
            components++;
            dfs(i);
        }
    }

    // 需要增加的点数 = 连通块数量 - 1
    cout << components - 1 << "\n";

    return 0;
}

## F 通配符匹配

In [ ]:
## add your code here
#include <iostream>
#include <string>
#include <vector>

using namespace std;

const int MAXN = 100005;

typedef unsigned long long ull;
const ull BASE = 13331; 
ull p_pow[MAXN];

void init_hash() {
    p_pow[0] = 1;
    for (int i = 1; i < MAXN; ++i) {
        p_pow[i] = p_pow[i-1] * BASE;
    }
}

ull hash_P[MAXN];
void build_hash_P(const string& P) {
    hash_P[0] = 0;
    for (int i = 0; i < P.length(); ++i) {
        hash_P[i+1] = hash_P[i] * BASE + P[i];
    }
}

ull get_hash_P(int L, int R) {
    return hash_P[R+1] - hash_P[L] * p_pow[R-L+1];
}

ull hash_S[MAXN];
void build_hash_S(const string& S) {
    hash_S[0] = 0;
    for (int i = 0; i < S.length(); ++i) {
        hash_S[i+1] = hash_S[i] * BASE + S[i];
    }
}

ull get_hash_S(int L, int R) {
    return hash_S[R+1] - hash_S[L] * p_pow[R-L+1];
}

struct Lit {
    int rel_start, rel_end; // 在当前块中的相对起点和终点
    int p_start, p_end;     // 在原模式串 P 中的绝对起点和终点
};

struct Block {
    int len;            // 整个块的长度（包含 '?')
    vector<Lit> lits;   // 块内的纯字母片段
};

bool is_match(const Block& blk, int s_idx) {
    for (const auto& lit : blk.lits) {
        int s_L = s_idx + lit.rel_start;
        int s_R = s_idx + lit.rel_end;
        // O(1) 终极比对大法
        if (get_hash_S(s_L, s_R) != get_hash_P(lit.p_start, lit.p_end)) {
            return false;
        }
    }
    return true;
}

int main() {
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);

    init_hash();

    string P;
    if (!(cin >> P)) return 0;

    build_hash_P(P);

    // 第 1 步：把模式串 P 按照 '*' 切开，得到每个块在 P 中的绝对区间 [start, end]
    vector<pair<int, int>> raw_blocks;
    int start = 0;
    for (int i = 0; i < P.length(); ++i) {
        if (P[i] == '*') {
            raw_blocks.push_back({start, i - 1});
            start = i + 1;
        }
    }
    raw_blocks.push_back({start, (int)P.length() - 1});

    // 第 2 步：精细化处理每个块，把里面的 '?' 剔除，提取纯字母片段
    vector<Block> parsed_blocks;
    for (auto b : raw_blocks) {
        Block blk;
        blk.len = b.second - b.first + 1;
        int l_start = -1;
        for (int i = b.first; i <= b.second; ++i) {
            if (P[i] != '?') {
                if (l_start == -1) l_start = i;
            } else {
                if (l_start != -1) {
                    blk.lits.push_back({l_start - b.first, i - 1 - b.first, l_start, i - 1});
                    l_start = -1;
                }
            }
        }
        if (l_start != -1) {
            blk.lits.push_back({l_start - b.first, b.second - b.first, l_start, b.second});
        }
        parsed_blocks.push_back(blk);
    }

    int n;
    if (!(cin >> n)) return 0;

    while (n--) {
        string S;
        cin >> S;

        // 特判：如果没有 '*'，只有唯一一个块，必须全等匹配
        if (parsed_blocks.size() == 1) {
            if (S.length() != parsed_blocks[0].len) {
                cout << "NO\n";
            } else {
                build_hash_S(S);
                if (is_match(parsed_blocks[0], 0)) cout << "YES\n";
                else cout << "NO\n";
            }
            continue;
        }

        // 长度底线校验：文件名的长度不能短于除了 '*' 之外的所有字符总长
        int min_required_len = 0;
        for (const auto& blk : parsed_blocks) min_required_len += blk.len;
        if (S.length() < min_required_len) {
            cout << "NO\n";
            continue;
        }

        build_hash_S(S);

        if (!is_match(parsed_blocks.front(), 0)) {
            cout << "NO\n";
            continue;
        }
        int curr_s = parsed_blocks.front().len;

        int end_s = S.length() - parsed_blocks.back().len;
        if (!is_match(parsed_blocks.back(), end_s)) {
            cout << "NO\n";
            continue;
        }

        // 贪心搜索中间的块
        bool overall_match = true;
        for (int i = 1; i < parsed_blocks.size() - 1; ++i) {
            const auto& blk = parsed_blocks[i];
            if (blk.len == 0) continue; // 遇到连续的 ** 产生的空块直接跳过
            
            bool found = false;
            // 只需要往后顺延寻找最早匹配的位置
            for (int s_idx = curr_s; s_idx <= end_s - blk.len; ++s_idx) {
                if (is_match(blk, s_idx)) {
                    found = true;
                    curr_s = s_idx + blk.len; // 贪心：找到就更新起点并立刻 break
                    break;
                }
            }
            if (!found) {
                overall_match = false;
                break;
            }
        }

        if (overall_match) cout << "YES\n";
        else cout << "NO\n";
    }

    return 0;
}

## G 汉诺塔

In [ ]:
## add your code here
#include <iostream>
#include <string>
#include <map>

using namespace std;

int main() {
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);
    
    int n;
    if (!(cin >> n)) return 0;
    
    map<string, int> priority;
    for (int i = 0; i < 6; i++) {
        string s;
        cin >> s;
        priority[s] = i; // 索引越小，优先级越高
    }
    
    // dest[i][x] 记录 i 个盘子从 x 柱子移动的终点 (0=A, 1=B, 2=C)
    int dest[35][3];
    // f[i][x] 记录对应的步数
    long long f[35][3];
    
    // 初始化 i = 1 的基础情况
    for (int i = 0; i < 3; i++) {
        char from = 'A' + i;
        char to1 = 'A' + (i + 1) % 3;
        char to2 = 'A' + (i + 2) % 3;
        
        string op1 = ""; op1 += from; op1 += to1;
        string op2 = ""; op2 += from; op2 += to2;
        
        if (priority[op1] < priority[op2]) {
            dest[1][i] = to1 - 'A';
        } else {
            dest[1][i] = to2 - 'A';
        }
        f[1][i] = 1;
    }
    
    // 递推 i >= 2 的情况
    for (int i = 2; i <= n; i++) {
        for (int x = 0; x < 3; x++) {
            int z = dest[i - 1][x];   // i-1 个盘子去的地方
            int y = 3 - x - z;        // 剩下的第三根柱子
            
            int w = dest[i - 1][z];   // i-1 个盘子从 z 继续移动会去的地方
            
            if (w == y) {
                // 情况 A
                dest[i][x] = y;
                f[i][x] = f[i - 1][x] + 1 + f[i - 1][z];
            } else {
                // 情况 B (w == x)
                dest[i][x] = z;
                f[i][x] = f[i - 1][x] + 1 + f[i - 1][z] + 1 + f[i - 1][x];
            }
        }
    }
    
    // 题目要求的是所有的盘子从柱子 A (索引 0) 开始移动所需的步数
    cout << f[n][0] << "\n";
    
    return 0;
}

## H 马步距离

In [ ]:
## add your code here
#include <iostream>
#include <cmath>
#include <algorithm>

using namespace std;

int main() {
    // 优化标准输入输出速度
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);

    long long xp, yp, xs, ys;
    
    // 读取输入直到 EOF
    while (cin >> xp >> yp >> xs >> ys) {
        // 取绝对值差并排序，保证 dx >= dy
        long long dx = abs(xp - xs);
        long long dy = abs(yp - ys);
        
        if (dx < dy) {
            swap(dx, dy);
        }

        // 处理特例
        if (dx == 1 && dy == 0) {
            cout << 3 << "\n";
            continue;
        }
        if (dx == 2 && dy == 2) {
            cout << 4 << "\n";
            continue;
        }

        // 计算基础步数限制，使用整数除法模拟向上取整
        // ceil(A / B) 在整数运算中等效于 (A + B - 1) / B
        long long res = max((dx + 1) / 2, (dx + dy + 2) / 3);

        // 奇偶性校验与修正
        if (res % 2 != (dx + dy) % 2) {
            res++;
        }

        cout << res << "\n";
    }

    return 0;
}

## I 直方图最大矩形

In [2]:
## add your code here
#include <vector>
#include <stack>
#include <algorithm>

using namespace std;
class Solution {
public:
    int largestRectangleArea(vector<int>& heights) {
        int maxArea = 0;
        stack<int> st;
        
        // 在末尾添加一个高度为0的哨兵，强制最后所有非零柱子出栈并计算
        heights.push_back(0); 
        
        for (int i = 0; i < heights.size(); ++i) {
            // 当遇到比栈顶矮的柱子时，开始计算以栈顶柱子为高的最大矩形
            while (!st.empty() && heights[i] < heights[st.top()]) {
                int h = heights[st.top()];
                st.pop();
                
                // 计算宽度：右边界为 i，左边界为弹出后的新栈顶
                int w = st.empty() ? i : i - st.top() - 1;
                
                // 更新最大面积
                maxArea = max(maxArea, h * w);
            }
            // 当前索引入栈，维持单调递增
            st.push(i);
        }
        
        // 恢复原数组结构（可选操作，避免对传入引用的数组造成永久修改）
        heights.pop_back(); 
        
        return maxArea;
    }
};

SyntaxError: invalid character '，' (U+FF0C) (1885484982.py, line 13)

## J 消防局的设立

In [1]:
## add your code here
#include <iostream>
#include <vector>
#include <algorithm>

using namespace std;

const int INF = 1e9;

int main() {
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);

    int n;
    if (!(cin >> n)) return 0;

    // parent[i] 记录节点 i 的父节点
    vector<int> parent(n + 1, 0);
    for (int i = 2; i <= n; i++) {
        cin >> parent[i];
    }

    // 初始化状态数组
    vector<int> dist_st(n + 1, INF); 
    vector<int> dist_un(n + 1, 0);   
    int ans = 0;

    // 自底向上逆序遍历
    for (int i = n; i >= 1; i--) {
        // 如果子树内的消防局足以覆盖子树内最远的未覆盖节点
        if (dist_st[i] + dist_un[i] <= 2) {
            dist_un[i] = -INF;
        }

        // 极限距离，必须在此处建站
        if (dist_un[i] == 2) {
            ans++;
            dist_st[i] = 0;
            dist_un[i] = -INF;
        }

        // 将当前节点状态传递给父节点
        if (i > 1) {
            int p = parent[i];
            dist_st[p] = min(dist_st[p], dist_st[i] + 1);
            dist_un[p] = max(dist_un[p], dist_un[i] + 1);
        }
    }

    // 检查根节点是否有遗漏
    if (dist_un[1] >= 0) {
        ans++;
    }

    cout << ans << "\n";

    return 0;
}

SyntaxError: invalid character '，' (U+FF0C) (2239377722.py, line 35)